*** DATA PREPROCESSING USING THE ORIGINAL DATASET ***

In [ ]:
import pandas as pd #importing pandas library
import numpy as np #Importing Numpuy library
import matplotlib.pyplot as plot #Importing the Matplot library

In [ ]:
#Loading the dataset into a dataframe
df= pd.read_csv('data/leaf_labelled_V1_cleaned_final.csv') #Reading the leaf data csv file in python
#Printing the dataframe
df

In [ ]:
df.info() #Summary of the basic information of the dataset

In [ ]:
df.isnull().sum() #Check for the total null value in the dataset

In [ ]:
print("The dimension of the leaf dataset is:", df.shape) #Shape of the dataset

In [ ]:
df.columns.values #view all the column names

In [ ]:
df.describe() #Summary statistics of all the numberic columns

*** PERFORMING SMOTE WITH PYTHON WITH THE ORIGINAL DATASET THEN SAVING IT IN ANOTHER DATAFRAME FOR FURTHER USE ***

In [ ]:
from imblearn.over_sampling import SMOTE

df2 = pd.read_csv('data/leaf_labelled_V1_cleaned_final.csv')

#Separate features and label
x = df2.drop("Class(Species)", axis=1)
y = df2["Class(Species)"]

class_counts = y.value_counts()
target_size = class_counts.max()
print("Target size per class is:", target_size)

samplying_strategy = {
    cls: target_size if count < target_size else count
    for cls, count in class_counts.items()
}

print("\n Sampling Strategy:")
print(samplying_strategy)

sm = SMOTE(sampling_strategy=samplying_strategy, k_neighbors=5)
x_resampled, y_resampled = sm.fit_resample(x,y)

df_balanced = pd.concat([x_resampled, y_resampled], axis=1)

print("\n New Class Distribution")
print(df_balanced["Class(Species)"].value_counts())

df_balanced.to_csv("Leaf_balanced_for_python.csv", index=False)
print("\n CSV File saved: Leaf_balanced_for_python.csv")

def save_arff(df2, filename, relation_name="leaf_balanced"):
    with open(filename, "w") as f:
        # Relation
        f.write(f"@relation {relation_name}\n\n")

        # Attributes
        for col in df2.columns:
            if col == "Class(Species)":
                classes = ",".join(sorted(df2[col].unique()))
                f.write(f"@attribute '{col}' {{{classes}}}\n")
            else:
                f.write(f"@attribute '{col}' numeric\n")
        
        # Data
        f.write("\n@data\n")
        for _, row in df2.iterrows():
            row_str = ",".join(str(val) for val in row)
            f.write(row_str + "\n")

# save ARFF
save_arff(df_balanced, "leaf_balanced_for_python.arff")

print("ARFF file saved: leaf_balanced_for_python.arff")

*** Modeling using Python with the balanced dataset using SMOTE to compare with the result I got from WEKA ***

In [ ]:
# Core liabraries
import pandas as pd
import numpy as np

# Modeling
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Classifiers
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# Metrics
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
# Class distribution
target_col = 'Class(Species)'
print("\nClass distribution:\n", df[target_col].value_counts())


X = df.drop(columns=[target_col])
y = df[target_col]

print("\nMissing values per column:\n", df.isnull().sum())



*** Models to be used ***

In [ ]:
models = {
    "kNN": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),
    "SVM (Linear)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel='linear', probability=False))
    ]),
    "Decision Tree": Pipeline([
        ("clf", DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=200, random_state=42))
    ]),
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GaussianNB())
    ])
}


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


*** Printing the Results from the models ***

In [ ]:

results = []

for name, pipe in models.items():
    # Fit
    pipe.fit(X_train, y_train)

    # Predict
    y_pred = pipe.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred, labels=np.unique(y))

    # Classification report
    report = classification_report(y_test, y_pred, digits=4)

    print(f"\n{'='*80}")
    print(f"{name} — Test Set Performance")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix (rows = true, cols = predicted):")
    print(pd.DataFrame(cm, index=np.unique(y), columns=np.unique(y)))
    print("\nClassification Report:")
    print(report)

    # Cross-validation (10-fold) on the whole dataset
    cv_scores = cross_val_score(pipe, X, y, cv=10, scoring='accuracy')
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()

    # Store summary for later table
    results.append({
        "Model": name,
        "Test Accuracy": acc,
        "CV Mean (10-fold)": cv_mean,
        "CV Std": cv_std
    })


*** Model Summary ***

In [ ]:

summary_df = pd.DataFrame(results).sort_values(by="CV Mean (10-fold)", ascending=False)
print("\n" + "="*80)
print("Model Comparison (sorted by 10-fold CV Mean Accuracy)")
print(summary_df.to_string(index=False,
                           formatters={
                               "Test Accuracy": lambda x: f"{x:.4f}",
                               "CV Mean (10-fold)": lambda x: f"{x:.4f}",
                               "CV Std": lambda x: f"{x:.4f}"
                           }))

*** Class Distribution using the Original Dataset ***

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.countplot(x='Class(Species)', data=df, palette='viridis')
plt.title('Class Distribution of Leaf Species')
plt.xticks(rotation=90)
plt.show()


*** Feature Correlation Heatmap using the Original Dataset ***

In [ ]:

plt.figure(figsize=(12,8))
sns.heatmap(df.drop('Class(Species)', axis=1).corr(), annot=False, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Prepare data
X = df.drop('Class(Species)', axis=1)
y = df['Class(Species)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

# Feature importance
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

*** Barplot using the Original Dataset ***

In [ ]:
# Ploting barplot for feature importance (Random Forest)
plt.figure(figsize=(10,6))
sns.barplot(x=importances, y=importances.index)
plt.title('Feature Importance (Random Forest)')
plt.show()

In [ ]:

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    "kNN": Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=5))]),
    "SVM": Pipeline([("scaler", StandardScaler()), ("clf", SVC(kernel='linear'))]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Naive Bayes": Pipeline([("scaler", StandardScaler()), ("clf", GaussianNB())])
}

results = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=10, scoring='accuracy')
    results.append({"Model": name, "Mean Accuracy": scores.mean(), "Std Dev": scores.std()})

summary_df = pd.DataFrame(results)
print(summary_df)
